# Uploading Models with KerasHub

**Author:** [Samaneh Saadat](https://github.com/SamanehSaadat/), [Matthew Watson](https://github.com/mattdangerw/)<br>
**Date created:** 2024/04/29<br>
**Last modified:** 2024/04/29<br>
**Description:** An introduction on how to upload a fine-tuned KerasHub model to model hubs.

# Introduction

Fine-tuning a machine learning model can yield impressive results for specific tasks.
Uploading your fine-tuned model to a model hub allows you to share it with the broader community.
By sharing your models, you'll enhance accessibility for other researchers and developers,
making your contributions an integral part of the machine learning landscape.
This can also streamline the integration of your model into real-world applications.

This guide walks you through how to upload your fine-tuned models to popular model hubs such as
[Kaggle Models](https://www.kaggle.com/models) and [Hugging Face Hub](https://huggingface.co/models).

# Setup

Let's start by installing and importing all the libraries we need. We use KerasHub for this guide.

In [1]:
!pip install -q --upgrade keras-hub huggingface-hub kagglehub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.3/731.3 kB 10.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
keras-nlp 0.18.1 requires keras-hub==0.18.1, but you have keras-hub 0.19.3 which is incompatible.


In [3]:
!pip uninstall -y keras-nlp keras-hub

Found existing installation: keras-nlp 0.18.1
Uninstalling keras-nlp-0.18.1:
  Successfully uninstalled keras-nlp-0.18.1
Found existing installation: keras-hub 0.19.3
Uninstalling keras-hub-0.19.3:
  Successfully uninstalled keras-hub-0.19.3


In [4]:
!pip install keras-hub

  Using cached keras_hub-0.19.3-py3-none-any.whl.metadata (7.7 kB)
Using cached keras_hub-0.19.3-py3-none-any.whl (731 kB)


In [5]:
import os

os.environ["KERAS_BACKEND"] = "jax"

import keras_hub


# Data

We can use the IMDB reviews dataset for this guide. Let's load the dataset from `tensorflow_dataset`.

In [6]:
import tensorflow_datasets as tfds

imdb_train, imdb_test = tfds.load(
    "imdb_reviews",
    split=["train", "test"],
    as_supervised=True,
    batch_size=4,
)

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/3 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.V8NWZF_1.0.0/imdb_reviews-train.tfrecor…

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.V8NWZF_1.0.0/imdb_reviews-test.tfrecord…

Generating unsupervised examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.V8NWZF_1.0.0/imdb_reviews-unsupervised.…

Dataset imdb_reviews downloaded and prepared to /root/tensorflow_datasets/imdb_reviews/plain_text/1.0.0. Subsequent calls will reuse this data.


We only use a small subset of the training samples to make the guide run faster.
However, if you need a higher quality model, consider using a larger number of training samples.

In [7]:
imdb_train = imdb_train.take(100)

# Task Upload

A `keras_hub.models.Task`, wraps a `keras_hub.models.Backbone` and a `keras_hub.models.Preprocessor` to create
a model that can be directly used for training, fine-tuning, and prediction for a given text problem.
In this section, we explain how to create a `Task`, fine-tune and upload it to a model hub.

## Load Model

If you want to build a Causal LM based on a base model, simply call `keras_hub.models.CausalLM.from_preset`
and pass a built-in preset identifier.

In [8]:
causal_lm = keras_hub.models.CausalLM.from_preset("gpt2_base_en")


100%|██████████| 431/431 [00:00<00:00, 1.01MB/s]


100%|██████████| 475M/475M [00:09<00:00, 53.8MB/s]


100%|██████████| 618/618 [00:00<00:00, 1.08MB/s]


100%|██████████| 0.99M/0.99M [00:00<00:00, 2.69MB/s]


100%|██████████| 446k/446k [00:00<00:00, 1.53MB/s]


## Fine-tune Model

After loading the model, you can call `.fit()` on the model to fine-tune it.
Here, we fine-tune the model on the IMDB reviews which makes the model movie domain-specific.

In [9]:
# Drop labels and keep the review text only for the Causal LM.
imdb_train_reviews = imdb_train.map(lambda x, y: x)

# Fine-tune the Causal LM.
causal_lm.fit(imdb_train_reviews)

100/100 ━━━━━━━━━━━━━━━━━━━━ 167s 1s/step - loss: 1.0205 - sparse_categorical_accuracy: 0.3294


## Save the Model Locally

To upload a model, you need to first save the model locally using `save_to_preset`.

In [10]:
preset_dir = "./gpt2_imdb"
causal_lm.save_to_preset(preset_dir)

Let's see the saved files.

In [11]:
os.listdir(preset_dir)

['model.weights.h5',
 'assets',
 'metadata.json',
 'tokenizer.json',
 'config.json',
 'task.json',
 'preprocessor.json']

### Load a Locally Saved Model

A model that is saved to a local preset can be loaded using `from_preset`.
What you save in, is what you get back out.

In [12]:
causal_lm = keras_hub.models.CausalLM.from_preset(preset_dir)

You can also load the `keras_hub.models.Backbone` and `keras_hub.models.Tokenizer` objects from this preset directory.
Note that these objects are equivalent to `causal_lm.backbone` and `causal_lm.preprocessor.tokenizer` above.

In [13]:
backbone = keras_hub.models.Backbone.from_preset(preset_dir)
tokenizer = keras_hub.models.Tokenizer.from_preset(preset_dir)

## Upload the Model to a Model Hub

After saving a preset to a directory, this directory can be uploaded to a model hub such as Kaggle or Hugging Face directly from the KerasHub library.
To upload the model to Kaggle, the URI must start with `kaggle://` and to upload to Hugging Face, it should start with `hf://`.

### Upload to Kaggle

To upload a model to Kaggle, first, we need to authenticate with Kaggle.
This can in one of the following ways:
1. Set environment variables `KAGGLE_USERNAME` and `KAGGLE_KEY`.
2. Provide a local `~/.kaggle/kaggle.json`.
3. Call `kagglehub.login()`.

Let's make sure we are logged in before continuing.

In [14]:
import kagglehub

if "KAGGLE_USERNAME" not in os.environ or "KAGGLE_KEY" not in os.environ:
    kagglehub.login()


Kaggle credentials set.
Kaggle credentials successfully validated.


To upload a model we can use `keras_hub.upload_preset(uri, preset_dir)` API where `uri` has the format of
`kaggle://<KAGGLE_USERNAME>/<MODEL>/Keras/<VARIATION>` for uploading to Kaggle and `preset_dir` is the directory that the model is saved in.

Running the following uploads the model that is saved in `preset_dir` to Kaggle:

In [15]:
kaggle_username = kagglehub.whoami()["username"]
kaggle_uri = f"kaggle://{kaggle_username}/gpt2/keras/gpt2_imdb"
keras_hub.upload_preset(kaggle_uri, preset_dir)

Kaggle credentials successfully validated.
Uploading Model https://www.kaggle.com/models/fedor57/gpt2/keras/gpt2_imdb ...
Model 'gpt2' does not exist or access is forbidden for user 'fedor57'. Creating or handling Model...
Model 'gpt2' Created.
Starting upload for file ./gpt2_imdb/model.weights.h5


Uploading: 100%|██████████| 498M/498M [00:04<00:00, 100MB/s]

Upload successful: ./gpt2_imdb/model.weights.h5 (475MB)
Starting upload for file ./gpt2_imdb/metadata.json



Uploading: 100%|██████████| 183/183 [00:00<00:00, 281B/s]

Upload successful: ./gpt2_imdb/metadata.json (183B)
Starting upload for file ./gpt2_imdb/tokenizer.json


...


Uploading: 100%|██████████| 456k/456k [00:00<00:00, 701kB/s]

Upload successful: ./gpt2_imdb/assets/tokenizer/merges.txt (446KB)
Starting upload for file ./gpt2_imdb/assets/tokenizer/vocabulary.json



Uploading: 100%|██████████| 1.04M/1.04M [00:00<00:00, 1.60MB/s]

Upload successful: ./gpt2_imdb/assets/tokenizer/vocabulary.json (1018KB)


Your model instance has been created.
Files are being processed...
See at: https://www.kaggle.com/models/fedor57/gpt2/keras/gpt2_imdb


### Upload to Hugging Face

To upload a model to Hugging Face, first, we need to authenticate with Hugging Face.
This can in one of the following ways:
1. Set environment variables `HF_USERNAME` and `HF_TOKEN`.
2. Call `huggingface_hub.notebook_login()`.

Let's make sure we are logged in before coninuing.

In [18]:
import huggingface_hub

if "HF_USERNAME" not in os.environ or "HF_TOKEN" not in os.environ:
    huggingface_hub.notebook_login()

`keras_hub.upload_preset(uri, preset_dir)` can be used to upload a model to Hugging Face if `uri` has the format of
`kaggle://<HF_USERNAME>/<MODEL>`.

Running the following uploads the model that is saved in `preset_dir` to Hugging Face:

In [17]:
hf_username = huggingface_hub.whoami()["name"]
hf_uri = f"hf://{hf_username}/gpt2_imdb"
keras_hub.upload_preset(hf_uri, preset_dir)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.weights.h5:   0%|          | 0.00/498M [00:00<?, ?B/s]

## Load a User Uploaded Model

After verifying that the model is uploaded to Kaggle, we can load the model by calling `from_preset`.

```python
causal_lm = keras_hub.models.CausalLM.from_preset(
    f"kaggle://{kaggle_username}/gpt2/keras/gpt2_imdb"
)
```

We can also load the model uploaded to Hugging Face by calling `from_preset`.

```python
causal_lm = keras_hub.models.CausalLM.from_preset(f"hf://{hf_username}/gpt2_imdb")
```

# Classifier Upload

Uploading a classifier model is similar to Causal LM upload.
To upload the fine-tuned model, first, the model should be saved to a local directory using `save_to_preset`
API and then it can be uploaded via `keras_hub.upload_preset`.

In [19]:
# Load the base model.
classifier = keras_hub.models.Classifier.from_preset(
    "bert_tiny_en_uncased", num_classes=2
)

# Fine-tune the classifier.
classifier.fit(imdb_train)

# Save the model to a local preset directory.
preset_dir = "./bert_tiny_imdb"
classifier.save_to_preset(preset_dir)

# Upload to Kaggle.
keras_hub.upload_preset(
    f"kaggle://{kaggle_username}/bert/keras/bert_tiny_imdb", preset_dir
)

100%|██████████| 454/454 [00:00<00:00, 888kB/s]


100%|██████████| 16.8M/16.8M [00:00<00:00, 23.6MB/s]


100%|██████████| 761/761 [00:00<00:00, 1.56MB/s]


100%|██████████| 226k/226k [00:00<00:00, 1.05MB/s]


100/100 ━━━━━━━━━━━━━━━━━━━━ 15s 70ms/step - loss: 0.6937 - sparse_categorical_accuracy: 0.5465
Uploading Model https://www.kaggle.com/models/fedor57/bert/keras/bert_tiny_imdb ...
Model 'bert' does not exist or access is forbidden for user 'fedor57'. Creating or handling Model...
Model 'bert' Created.
Starting upload for file ./bert_tiny_imdb/model.weights.h5


...


Uploading: 100%|██████████| 52.8M/52.8M [00:01<00:00, 44.0MB/s]

Upload successful: ./bert_tiny_imdb/task.weights.h5 (50MB)
Starting upload for file ./bert_tiny_imdb/assets/tokenizer/vocabulary.txt



Uploading: 100%|██████████| 232k/232k [00:00<00:00, 360kB/s]

Upload successful: ./bert_tiny_imdb/assets/tokenizer/vocabulary.txt (226KB)


Your model instance has been created.
Files are being processed...
See at: https://www.kaggle.com/models/fedor57/bert/keras/bert_tiny_imdb


After verifying that the model is uploaded to Kaggle, we can load the model by calling `from_preset`.

```python
classifier = keras_hub.models.Classifier.from_preset(
    f"kaggle://{kaggle_username}/bert/keras/bert_tiny_imdb"
)
```